# CELL 1

In [1]:
import os
os.environ["HADOOP_HOME"] = r"D:\hadoop"
os.environ["PATH"] = r"D:\hadoop\bin" + os.pathsep + os.environ["PATH"]

print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

HADOOP_HOME: D:\hadoop


In [2]:
%pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("capstone") \
    .master("local[*]") \
    .getOrCreate()

print("Cores available:", spark.sparkContext.defaultParallelism)



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Cores available: 8


# My Machine: 8 crores

# Cell 2

In [3]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("ts", StringType(), True),
    StructField("country", StringType(), True),
])

orders_raw = spark.read.csv("data/orders.csv", header=True, schema=orders_schema)
customers_raw = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)

orders_raw.printSchema()
orders_raw.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- ts: string (nullable = true)
 |-- country: string (nullable = true)

+--------+-----------+----------+------+-------------------+-------+
|order_id|customer_id|product_id|amount|                 ts|country|
+--------+-----------+----------+------+-------------------+-------+
|       1|       9750|       259|144.97|2024-12-02 15:28:59|     GB|
|       2|       9622|       359| 83.38|2024-11-26 02:40:48|     AU|
|       3|       1260|       303| 24.75|2024-07-06 22:28:43|     NP|
|       4|       5420|        77| 25.15|2024-01-13 06:03:50|     NP|
|       5|       9753|       431| 27.11|2023-05-09 22:52:00|     NP|
+--------+-----------+----------+------+-------------------+-------+
only showing top 5 rows


# Part A: Clean the Data

In [4]:
start_count = orders_raw.count()
print("Starting order rows:", start_count)

Starting order rows: 1020025


In [5]:
# 1
orders_cleaned = orders_raw.dropDuplicates(['order_id'])    # Duplicates on order_id only because order_id is a unique business key

after_cleaned_count = orders_cleaned.count()
duplicates_removed = start_count - after_cleaned_count
print("Duplicate rows removed:", duplicates_removed)

Duplicate rows removed: 20025


In [6]:
# 2
# Checks the null value or value that is 0 or negative and counts
bad_amount_count = orders_cleaned.filter(
    (F.col("amount").isNull()) | (F.col("amount") <= 0)
).count()
print("Bad amount rows (missing, zero, or negative):", bad_amount_count)

orders_amount_clean = orders_cleaned.filter(
    (F.col("amount").isNotNull()) & (F.col("amount") > 0)
)

Bad amount rows (missing, zero, or negative): 15022


In [7]:
# 3
orders_ts = orders_amount_clean.withColumn(
    "ts_parsed", F.try_to_timestamp(F.col("ts"), F.lit("yyyy-MM-dd HH:mm:ss"))
)

bad_ts_count = orders_ts.filter(F.col("ts_parsed").isNull()).count()
print("Rows with unparseable timestamp:", bad_ts_count)

orders_ts_clean = orders_ts.filter(F.col("ts_parsed").isNotNull())

Rows with unparseable timestamp: 4859


In [8]:
# 4 deduplicate customers, keep latest signup_date
latest_signup = customers_raw.groupBy("customer_id").agg(F.max("signup_date").alias(("latest_signup_date")))

ls = latest_signup.alias("ls")
cr = customers_raw.alias("cr")

clean_customers = ls.join(
    cr,
    on=(F.col("ls.customer_id") == F.col("cr.customer_id")) &
       (F.col("ls.latest_signup_date") == F.col("cr.signup_date")),
    how="inner"
).select(
    F.col("cr.customer_id"),
    F.col("cr.name"),
    F.col("cr.country"),
    F.col("cr.signup_date")
)


before_customer_count = customers_raw.count()
after_customer_count = clean_customers.count()
print("Customer rows before dedup:", before_customer_count)
print("Customer rows after dedup:", after_customer_count)


Customer rows before dedup: 10300
Customer rows after dedup: 10000


In [9]:
# 5 find orphan orders
orphan_orders = orders_ts_clean.join(clean_customers, on="customer_id", how="left_anti")
orphan_count = orphan_orders.count()
print("Orphans orders:", orphan_count)

clean_orders = orders_ts_clean.join(clean_customers, on="customer_id", how="inner").select(orders_ts_clean["*"])

clean_order_count = clean_orders.count()
print("Final clean orders:", clean_order_count)

orphan_orders.coalesce(1).write.csv("out/orphan_orders", header=True, mode="overwrite")
# coalesce(1) safe here: orphan_orders is a small output, so merging it into one file for readability won't cause a memory or performance problem.

Orphans orders: 9892
Final clean orders: 970227


In [10]:
# 6 full accounting report

print("=== Part A: Data Cleaning Accounting Report ===")
print("Orders — starting rows:", start_count)
print("Orders — duplicates removed:", duplicates_removed)
print("Orders — bad amounts removed:", bad_amount_count)
print("Orders — bad timestamps removed:", bad_ts_count)
print("Orders — orphans set aside:", orphan_count)
print("Orders — final clean rows:", clean_order_count)
print()
print("Customers — starting rows:", before_customer_count)
print("Customers — after dedup (final clean rows):", after_customer_count)

=== Part A: Data Cleaning Accounting Report ===
Orders — starting rows: 1020025
Orders — duplicates removed: 20025
Orders — bad amounts removed: 15022
Orders — bad timestamps removed: 4859
Orders — orphans set aside: 9892
Orders — final clean rows: 970227

Customers — starting rows: 10300
Customers — after dedup (final clean rows): 10000
